# 3-Year 95% CVaR with a D-vine Copula
This notebook implements the requested pipeline for quarterly returns using `polars` for data handling and simulation, `altair` for visualizations, and a D-vine copula for cross-sectional dependence.

Pipeline:
1. Load the provided quarterly return dataset.
2. Optionally unsmooth private-asset returns if you are starting from appraisal-smoothed series.
3. Fit ARMA(1,1)-GARCH(1,1) marginals with either skewed-t or stable standardized residuals.
4. Fit a D-vine copula to the PIT residuals.
5. Simulate 3-year quarterly paths and estimate 95% CVaR.

If `pyvinecopulib` is not installed yet, install it before running this notebook.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import warnings

import altair as alt
import numpy as np
import pandas as pd
import polars as pl
from arch import arch_model
from scipy.stats import levy_stable
from statsmodels.tsa.arima.model import ARIMA
from sklearn.linear_model import LinearRegression

try:
    import pyvinecopulib as pv
except ImportError as exc:
    pv = None
    PYVINECOPULIB_IMPORT_ERROR = exc

warnings.filterwarnings('ignore')
alt.data_transformers.disable_max_rows()
pd.set_option('display.max_columns', None)

DATA_PATH = Path('../data/processed_returns.csv')
DATE_COLUMN = 'date'
HORIZON_QUARTERS = 12
N_SIMULATIONS = 50_000
ALPHA = 0.05
RESIDUAL_FAMILY = 'skewt'  # set to 'stable' to use a stable residual distribution
APPLY_UNSMOOTHING = False  # keep False for the provided processed dataset
PRIVATE_ASSETS = {'PE', 'NPI'}
PORTFOLIO_WEIGHTS = {'SPY': 0.40, 'AGG': 0.20, 'PE': 0.25, 'NPI': 0.15}
EXPECTED_RETURNS = {'SPY': (1+0.06)**(1/4) - 1, 'AGG': (1+0.045)**(1/4) - 1, 'PE': (1+0.095)**(1/4) - 1, 'NPI': (1+0.07)**(1/4) - 1}
D_VINE_ORDER = ['SPY', 'AGG', 'PE', 'NPI']
RNG_SEED = 7

print('Configuration loaded.')

Configuration loaded.


In [2]:
ret_df = pd.read_csv('../data/processed_returns.csv', index_col=0, parse_dates=True)
ret_df = ret_df/100
ret_df = np.log(1 + ret_df)
ret_df.index.name = DATE_COLUMN
asset_cols = [column for column in D_VINE_ORDER if column in ret_df.columns]
if len(asset_cols) != 4:
    asset_cols = [column for column in ret_df.columns if pd.api.types.is_numeric_dtype(ret_df[column])]
    asset_cols = list(asset_cols)

if APPLY_UNSMOOTHING:
    print('Applying Geltner-style unsmoothing to private assets.')
    for asset in PRIVATE_ASSETS.intersection(asset_cols):
        values = ret_df[asset].to_numpy(dtype=float)
        if len(values) > 1:
            model = LinearRegression().fit(values[:-1].reshape(-1, 1), values[1:])
            alpha = model.coef_[0]
            alpha = float(np.clip(alpha, 0.0, 0.95))
            unsmoothed = values.copy()
            unsmoothed[0] = values[0]
            
            unsmoothed[1:] = (values[1:] - alpha * values[:-1]) / (1.0 - alpha)
            
            smoothed_ann_vol = np.std(values) * np.sqrt(4)
            unsmoothed_ann_vol = np.std(unsmoothed) * np.sqrt(4)
            
            print(f'Asset: {asset}, alpha: {alpha:.4f}, Smoothed Annual Vol: {smoothed_ann_vol:.4f}, Unsmoothed Annual Vol: {unsmoothed_ann_vol:.4f}')
            
            ret_df[asset] = unsmoothed

model_df = ret_df[asset_cols].dropna().copy()
ret_pl = pl.from_pandas(model_df.reset_index())
returns_long = ret_pl.unpivot(index=DATE_COLUMN, on=asset_cols, variable_name='asset', value_name='return')

print(f'Raw shape: {ret_df.shape}')
print(f'Modeling shape: {model_df.shape}')
print('Assets:', asset_cols)
ret_pl.head()

Raw shape: (96, 4)
Modeling shape: (81, 4)
Assets: ['SPY', 'AGG', 'PE', 'NPI']


date,SPY,AGG,PE,NPI
datetime[μs],f64,f64,f64,f64
2003-12-31 00:00:00,0.112094,0.003692,0.094241,0.034574
2004-03-31 00:00:00,0.019722,0.02256,-0.044602,0.092938
2004-06-30 00:00:00,0.016208,-0.024996,0.076009,0.062427
2004-09-30 00:00:00,-0.020329,0.030419,0.064438,0.060961
2004-12-31 00:00:00,0.086034,0.009152,0.039644,0.049436


In [3]:
def fit_arma_garch_marginal(series: pd.Series, residual_family: str = 'skewt') -> dict:
    values = series.dropna().astype(float).to_numpy()
    if values.size < 12:
        raise ValueError(f'{series.name} has too few observations for ARMA-GARCH fitting.')

    garch_dist = 'skewt' if residual_family == 'skewt' else 't'
    garch_fit = arch_model(
        values, # original: arma_resid
        mean='AR', # change to Z to AR
        lags=1,
        vol='GARCH',
        p=1,
        q=1,
        dist=garch_dist,
        rescale=False,
    ).fit(disp='off')

    std_resid_raw = garch_fit.std_resid
    if hasattr(std_resid_raw, 'dropna'):
        std_resid_raw = std_resid_raw.dropna()
    std_resid = np.asarray(std_resid_raw, dtype=float)
    eps = 1e-10

    if residual_family == 'skewt':
        from arch.univariate import SkewStudent

        dist = SkewStudent()
        eta = float(garch_fit.params['eta'])
        lam = float(garch_fit.params['lambda'])
        dist_params = np.array([eta, lam], dtype=float)
        u = np.asarray(dist.cdf(std_resid, dist_params), dtype=float)

        def ppf_fn(q: np.ndarray) -> np.ndarray:
            q = np.clip(np.asarray(q, dtype=float), eps, 1.0 - eps)
            return np.asarray(dist.ppf(q, dist_params), dtype=float)

        dist_summary = {'family': 'skewt', 'eta': eta, 'lambda': lam}
    elif residual_family == 'stable':
        stable_params = levy_stable.fit(std_resid)
        u = np.asarray(levy_stable.cdf(std_resid, *stable_params), dtype=float)

        def ppf_fn(q: np.ndarray) -> np.ndarray:
            q = np.clip(np.asarray(q, dtype=float), eps, 1.0 - eps)
            return np.asarray(levy_stable.ppf(q, *stable_params), dtype=float)

        dist_summary = {
            'family': 'stable',
            'stable_alpha': float(stable_params[0]),
            'stable_beta': float(stable_params[1]),
            'stable_loc': float(stable_params[2]),
            'stable_scale': float(stable_params[3]),
        }
    else:
        raise ValueError('residual_family must be either skewt or stable.')

    u = np.clip(u, eps, 1.0 - eps)

    params = garch_fit.params
    conditional_volatility = np.asarray(garch_fit.conditional_volatility, dtype=float)
    model_summary = {
        'Asset': series.name,
        'N': int(values.size),
        'GARCH_AIC': float(garch_fit.aic),
        'GARCH_BIC': float(garch_fit.bic),
        'Const':float(params.get('Const', np.nan)),
        'Phi':float(params.get('y[1]', np.nan)),
        'Omega': float(params.get('omega', np.nan)),
        'Alpha': float(params.get('alpha[1]', np.nan)),
        'Beta': float(params.get('beta[1]', np.nan)),
        'Persistence': float(params.get('alpha[1]', 0.0) + params.get('beta[1]', 0.0)),
    }
    model_summary.update(dist_summary)

    return {
        'asset': series.name,
        'observed': values,
        'garch_fit': garch_fit,
        'std_resid': std_resid,
        'last_std_resid': float(std_resid[-1]),
        'pit': u,
        'ppf': ppf_fn,
        'model_summary': model_summary,
        'last_return': float(values[-1]),
        'last_sigma': float(conditional_volatility[-1]),
        'const':float(params.get('Const', np.nan)),
        'phi':float(params.get('y[1]', np.nan)),
        'omega': float(params.get('omega', np.nan)),
        'alpha': float(params.get('alpha[1]', 0.0)),
        'beta': float(params.get('beta[1]', 0.0)),
    }


marginals = [fit_arma_garch_marginal(model_df[column], residual_family=RESIDUAL_FAMILY) for column in asset_cols]
marginal_summary = pl.DataFrame([fit['model_summary'] for fit in marginals])
marginal_summary

/tmp/ipykernel_81817/4215625651.py:16: ConvergenceWarning: The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.

  ).fit(disp='off')


Asset,N,GARCH_AIC,GARCH_BIC,Const,Phi,Omega,Alpha,Beta,Persistence,family,eta,lambda
str,i64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64
"""SPY""",81,-211.66744,-194.993253,0.017802,-0.133038,0.000981,0.267076,0.699109,0.966185,"""skewt""",4.00911,-0.883117
"""AGG""",81,-371.256755,-354.582568,0.008192,-0.006252,0.000037,0.101563,0.842847,0.944411,"""skewt""",8.036798,-0.189987
"""PE""",81,-183.350627,-166.676441,0.029301,0.125566,0.001185,0.733934,0.266066,1.0,"""skewt""",3.357618,-0.319211
"""NPI""",81,-237.583639,-220.909453,0.010158,0.306444,0.000225,0.589544,0.410456,1.0,"""skewt""",3.169983,-0.523367


In [4]:
marginals[0]['garch_fit']

                                 AR - GARCH Model Results                                
Dep. Variable:                                 y   R-squared:                      -0.038
Mean Model:                                   AR   Adj. R-squared:                 -0.052
Vol Model:                                 GARCH   Log-Likelihood:                112.834
Distribution:      Standardized Skew Student's t   AIC:                          -211.667
Method:                       Maximum Likelihood   BIC:                          -194.993
                                                   No. Observations:                   80
Date:                           Wed, Apr 29 2026   Df Residuals:                       78
Time:                                   16:44:01   Df Model:                            2
                                  Mean Model                                 
                 coef    std err          t      P>|t|       95.0% Conf. Int.
----------------------------------

In [5]:
def fit_rvine_copula(marginals: list[dict]) -> dict:
    u = np.column_stack([fit['pit'] for fit in marginals])
    controls = pv.FitControlsVinecop(
        family_set=[pv.student, pv.clayton, pv.gumbel, pv.frank, pv.gaussian],
        selection_criterion='bic',
        preselect_families=True,
        tree_criterion='tau',
    )
    copula = pv.Vinecop.from_data(data=u, controls=controls)

    copula_rows = []
    for tree_idx, family_row in enumerate(copula.families, start=1):
        tau_row = copula.taus[tree_idx - 1]
        for edge_idx, family in enumerate(family_row, start=1):
            copula_rows.append({
                'Tree': tree_idx,
                'Edge': edge_idx,
                'Family': str(family),
                'Tau': float(tau_row[edge_idx - 1]),
            })

    return {
        'copula': copula,
        'u': u,
        'summary': pl.DataFrame(copula_rows),
    }


vine_result = fit_rvine_copula(marginals)
rvine_copula = vine_result['copula']
copula_summary = vine_result['summary']
copula_summary

Tree,Edge,Family,Tau
i64,i64,str,f64
1,1,"""BicopFamily.clayton""",-0.115719
1,2,"""BicopFamily.clayton""",0.214425
1,3,"""BicopFamily.student""",0.16526
2,1,"""BicopFamily.gumbel""",0.05658
2,2,"""BicopFamily.gumbel""",-0.050897
3,1,"""BicopFamily.gaussian""",-0.024171


In [6]:
rvine_copula

<pyvinecopulib.Vinecop> Vinecop model with 4 variables
tree edge conditioned variables conditioning variables var_types   family rotation parameters  df   tau 
   1    1                  2, 1                             c, c  Clayton       90       0.26 1.0 -0.12 
   1    2                  1, 3                             c, c  Clayton        0       0.55 1.0  0.21 
   1    3                  3, 4                             c, c  Student        0 0.26, 2.66 2.0  0.17 
   2    1                  2, 3                      1      c, c   Gumbel      180       1.06 1.0  0.06 
   2    2                  1, 4                      3      c, c   Gumbel      270       1.05 1.0 -0.05 
   3    1                  2, 4                   3, 1      c, c Gaussian        0      -0.04 1.0 -0.02 

In [7]:
marginals[0]

{'asset': 'SPY',
 'observed': array([ 0.11209384,  0.0197218 ,  0.0162083 , -0.02032868,  0.08603365,
        -0.02045033,  0.01431755,  0.03612275,  0.01716313,  0.04580889,
        -0.01542634,  0.05269919,  0.06400313,  0.00663804,  0.06197116,
         0.01892429, -0.03735146, -0.09753094, -0.02571133, -0.09259832,
        -0.24294631, -0.11933246,  0.15086954,  0.14307041,  0.05929218,
         0.05279686, -0.12056474,  0.10581469,  0.10220302,  0.0573006 ,
         0.00025868, -0.14869995,  0.10991319,  0.11947863, -0.02885324,
         0.0615388 , -0.00382729,  0.09986961,  0.02893079,  0.05109071,
         0.10006953,  0.01688714,  0.05030319,  0.01128356,  0.04783967,
         0.0087747 ,  0.00203851, -0.06639575,  0.0678501 ,  0.01320477,
         0.02427203,  0.03707697,  0.03875589,  0.05752488,  0.03024626,
         0.04320848,  0.06545332, -0.01000266,  0.03490757,  0.07372567,
        -0.14539685,  0.12686203,  0.04141652,  0.01739189,  0.08606399,
        -0.21626343,  

In [9]:
def simulate_3y_paths(marginals: list[dict], copula, n_simulations: int, horizon_quarters: int, seed: int = 7) -> dict:
    d = len(marginals)
    returns = np.zeros((n_simulations, horizon_quarters, d), dtype=float)

    prev_return = np.column_stack([np.full(n_simulations, fit['last_return'], dtype=float) for fit in marginals])
    prev_eps = np.column_stack([np.full(n_simulations, fit['last_std_resid'], dtype=float) for fit in marginals])
    prev_sigma = np.column_stack([np.full(n_simulations, max(fit['last_sigma'], 1e-8), dtype=float) for fit in marginals])

    for quarter in range(horizon_quarters):
        u = copula.simulate(n_simulations, seeds=[seed + quarter])
        for idx, fit in enumerate(marginals):
            z = fit['ppf'](u[:, idx])
            sigma2 = fit['omega'] + fit['alpha'] * np.square(prev_eps[:, idx]) + fit['beta'] * np.square(prev_sigma[:, idx])
            sigma = np.sqrt(np.maximum(sigma2, 1e-12))
            mean = fit['const'] + fit['phi'] * prev_return[:, idx] # EXPECTED_RETURNS[fit['asset']]
            eps = sigma * z
            r = mean + eps
            r = np.exp(r) - 1.0
            r = np.clip(r, -0.9999, 1.0)
            returns[:, quarter, idx] = r
            prev_return[:, idx] = r
            prev_eps[:, idx] = eps
            prev_sigma[:, idx] = sigma

    weights = np.array([PORTFOLIO_WEIGHTS[fit['asset']] for fit in marginals], dtype=float)
    quarter_portfolio_pct = np.einsum('qij,j->qi', returns, weights)
    horizon_compounded_return = np.prod(1.0 + quarter_portfolio_pct, axis=1) - 1.0
    horizon_asset_linear = np.sum(returns * weights.reshape(1, 1, -1), axis=1)

    return {
        'returns': returns,
        'weights': weights,
        'quarter_portfolio': quarter_portfolio_pct,
        'compounded_return': horizon_compounded_return,
        'asset_contrib_simple': horizon_asset_linear,
    }


simulation = simulate_3y_paths(marginals, rvine_copula, N_SIMULATIONS, HORIZON_QUARTERS, seed=RNG_SEED)
simulation_pl = pl.DataFrame({
    'compounded_return': simulation['compounded_return'],
})
simulation_pl.head()

compounded_return
f64
1.660748
1.218226
0.927479
0.441917
1.410243


In [10]:
q_5 = simulation_pl.select(pl.col('compounded_return').quantile(ALPHA)).item()
cvar_95 = simulation_pl.filter(pl.col('compounded_return') <= q_5).select(pl.col('compounded_return').mean()).item()
var_95 = q_5
mean_return = simulation_pl.select(pl.col('compounded_return').mean()).item()
std_return = simulation_pl.select(pl.col('compounded_return').std()).item()
tail_mean = simulation_pl.filter(pl.col('compounded_return') <= q_5).select(pl.col('compounded_return').mean()).item()

component_rows = []
tail_mask = simulation['compounded_return'] <= q_5
for idx, fit in enumerate(marginals):
    component = -simulation['asset_contrib_simple'][tail_mask, idx].mean()
    component_rows.append({
        'Asset': fit['asset'],
        'ComponentCVaR': float(component),
        'ShareOfCVaR': float(component / cvar_95),
    })
component_summary = pl.DataFrame(component_rows)

risk_summary = pl.DataFrame([
    {'Metric': 'Mean 3Y Return', 'Value': mean_return},
    {'Metric': 'Std Dev', 'Value': std_return},
    {'Metric': 'VaR 95%', 'Value': var_95},
    {'Metric': 'CVaR 95%', 'Value': cvar_95},
])

risk_summary

Metric,Value
str,f64
"""Mean 3Y Return""",1.099132
"""Std Dev""",1.130414
"""VaR 95%""",-0.2492
"""CVaR 95%""",-0.462234


In [11]:
asset_chart = (
    alt.Chart(pd.DataFrame(returns_long.to_dict(as_series=False)))
    .mark_line()
    .encode(
        x=alt.X(f'{DATE_COLUMN}:T', title='Date'),
        y=alt.Y('return:Q', title='Quarterly return'),
        color=alt.Color('asset:N', title='Asset'),
    )
    .properties(title='Quarterly Returns by Asset', width=820, height=320)
)

distribution_chart = (
    alt.Chart(pd.DataFrame(simulation_pl.to_dict(as_series=False)))
    .transform_bin('bin', 'compounded_return', bin=alt.Bin(maxbins=150))
    .transform_aggregate(count='count()', groupby=['bin'])
    .mark_bar(color='#2a9d8f')
    .encode(
        x=alt.X('bin:Q', title='3-year compounded return'),
        y=alt.Y('count:Q', title='Frequency'),
    )
    .properties(title='Simulated 3-Year Portfolio Return Distribution', width=820, height=300)
)

tail_rule = alt.Chart(pd.DataFrame({'x': [q_5], 'label': ['VaR 5%']})).mark_rule(color='crimson', strokeDash=[6, 4]).encode(x='x:Q')
cvar_rule = alt.Chart(pd.DataFrame({'x': [tail_mean], 'label': ['Tail Mean']})).mark_rule(color='black').encode(x='x:Q')

risk_bar = (
    alt.Chart(pd.DataFrame(component_summary.to_dict(as_series=False)))
    .mark_bar()
    .encode(
        x=alt.X('Asset:N', title='Asset'),
        y=alt.Y('ComponentCVaR:Q', title='CVaR contribution'),
        color=alt.Color('Asset:N', legend=None),
        tooltip=['Asset', 'ComponentCVaR', 'ShareOfCVaR'],
    )
    .properties(title='Component CVaR Contributions', width=520, height=300)
)

asset_chart

alt.Chart(...)

In [12]:
distribution_chart_with_rules = distribution_chart + tail_rule + cvar_rule
distribution_chart_with_rules

alt.LayerChart(...)

In [13]:
risk_bar

alt.Chart(...)

## Notes
- The notebook keeps the provided CSV load line intact and does not modify any existing project file.
- `RESIDUAL_FAMILY` can be switched between `skewt` and `stable`.
- If you want to fit on appraisal-smoothed private returns instead of the already-processed dataset, set `APPLY_UNSMOOTHING = True`.
- The CVaR decomposition below uses the linearized portfolio return in decimal units, while the reported portfolio CVaR is based on the 3-year compounded return.